In [ ]:
# Embedded from Playground_Kai/checkpoints/best_hyperparams_rnn_recons_raw_backup.yaml
BEST_HYPERPARAMS_YAML = """
lr: 0.0008215117598353862
weight_decay: 0.00935611311371853
proj_dim: 64
hidden_size: 256
num_layers: 1
dropout: 0.11750003567529757
trial_val_cer: 98.469
pipeline: recons-raw
search_mode: two-phase
search_phase_found: F1
num_coarse_trials: 20
num_fine_trials: 10
coarse_epochs: 12
fine_epochs: 12
num_trial_sessions: 8
fine_shrink: 3.0
tuned_at: '2026-03-12 17:31:07'
"""

# Embedded from config/user/single_user.yaml
SPLIT_YAML = """
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626917264-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622889105-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627001995-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626915176-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  val:
  - user: 89335547
    session: 2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  test:
  - user: 89335547
    session: 2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
"""


# RNN Recons Raw (Self-Contained)

This notebook reproduces the `Playground_Kai.train_recons` RNN run intent:
`python -m Playground_Kai.train_recons --model rnn --from-hyperparams .../best_hyperparams_rnn_recons_raw_backup.yaml --epochs 150`

All required model/data/decoder/metrics/logger logic is inlined.

Expected external assets:
- Recons files: `data_recons/*_recons_v3.hdf5`
- Installed dependencies from `Playground_Kai/requirements.txt`


In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
import string
import time
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Sequence

import h5py
import Levenshtein
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def seed_everything(seed: int = 13) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CharacterSet:
    def __init__(self) -> None:
        chars = list(string.ascii_letters + string.digits + string.punctuation)
        special = ["Key.backspace", "Key.enter", "Key.space", "Key.shift"]
        self.allowed_keys = chars + special
        self.key_to_label = {k: i for i, k in enumerate(self.allowed_keys)}
        self.label_to_key = {i: k for k, i in self.key_to_label.items()}
        self.null_class = len(self.allowed_keys)
        self.num_classes = len(self.allowed_keys) + 1

    def labels_to_text(self, labels: Sequence[int]) -> str:
        out: list[str] = []
        for raw in labels:
            i = int(raw)
            if i not in self.label_to_key:
                continue
            key = self.label_to_key[i]
            if key == "Key.backspace":
                out.append("\b")
            elif key == "Key.enter":
                out.append("\n")
            elif key == "Key.space":
                out.append(" ")
            elif key == "Key.shift":
                out.append("\x11")
            else:
                out.append(key)
        return "".join(out)


CHARSET = CharacterSet()


@dataclass
class LabelData:
    text: str

    @classmethod
    def from_labels(cls, labels: Sequence[int]) -> "LabelData":
        return cls(text=CHARSET.labels_to_text(labels))

    @classmethod
    def from_keystrokes(cls, keystrokes: Sequence[dict[str, Any]], start_t: float, end_t: float) -> "LabelData":
        mapped: list[int] = []
        for ks in keystrokes:
            if ks["start"] > end_t:
                break
            if ks["start"] < start_t:
                continue
            key = ks["key"]
            if key in CHARSET.key_to_label:
                mapped.append(CHARSET.key_to_label[key])
        return cls.from_labels(mapped)

    @property
    def labels(self) -> np.ndarray:
        keys: list[str] = []
        for c in self.text:
            if c == "\b":
                keys.append("Key.backspace")
            elif c == "\n":
                keys.append("Key.enter")
            elif c == " ":
                keys.append("Key.space")
            elif c == "\x11":
                keys.append("Key.shift")
            else:
                keys.append(c)
        out = [CHARSET.key_to_label[k] for k in keys if k in CHARSET.key_to_label]
        return np.asarray(out, dtype=np.int32)

    def __len__(self) -> int:
        return len(self.text)


class CTCGreedyDecoder:
    def decode_batch(self, emissions: np.ndarray, emission_lengths: np.ndarray) -> list[LabelData]:
        out: list[LabelData] = []
        for b in range(emissions.shape[1]):
            seq = emissions[: int(emission_lengths[b]), b]
            prev = CHARSET.null_class
            decoded: list[int] = []
            for y in seq.argmax(-1):
                yi = int(y)
                if yi != CHARSET.null_class and yi != prev:
                    decoded.append(yi)
                prev = yi
            out.append(LabelData.from_labels(decoded))
        return out


class CharacterErrorRates:
    def __init__(self) -> None:
        self.ins = 0
        self.del_ = 0
        self.sub = 0
        self.tgt = 0

    def update(self, prediction: LabelData, target: LabelData) -> None:
        edits = Counter(op for op, _, _ in Levenshtein.editops(prediction.text, target.text))
        self.ins += int(edits["insert"])
        self.del_ += int(edits["delete"])
        self.sub += int(edits["replace"])
        self.tgt += len(target)

    def compute(self) -> dict[str, float]:
        d = max(self.tgt, 1)
        return {
            "CER": (self.ins + self.del_ + self.sub) / d * 100.0,
            "IER": self.ins / d * 100.0,
            "DER": self.del_ / d * 100.0,
            "SER": self.sub / d * 100.0,
        }


class ReconsRawDataset(Dataset):
    def __init__(self, hdf5_path: Path, window_length: int = 500, stride: int | None = None, padding: tuple[int, int] = (10, 2), jitter: bool = False) -> None:
        self.hdf5_path = Path(hdf5_path)
        self.window_length = int(window_length)
        self.stride = int(stride) if stride is not None else int(window_length)
        self.left_padding, self.right_padding = padding
        self.jitter = jitter

        with h5py.File(self.hdf5_path, "r") as f:
            grp = f["emg2qwerty"]
            self.session_length = int(grp["timeseries/emg_left"].shape[0])
            self.keystrokes = json.loads(grp.attrs.get("keystrokes", "[]"))

        self._file: h5py.File | None = None

    def _ensure_open(self) -> None:
        if self._file is None:
            self._file = h5py.File(self.hdf5_path, "r")

    def __len__(self) -> int:
        return int(max(self.session_length - self.window_length, 0) // self.stride + 1)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        self._ensure_open()
        assert self._file is not None
        ts = self._file["emg2qwerty/timeseries"]

        offset = idx * self.stride
        leftover = self.session_length - (offset + self.window_length)
        if leftover < 0:
            raise IndexError
        if leftover > 0 and self.jitter:
            offset += int(np.random.randint(0, min(self.stride, leftover)))

        window_start = max(offset - self.left_padding, 0)
        window_end = min(offset + self.window_length + self.right_padding, self.session_length)

        emg_left = ts["emg_left"][window_start:window_end]
        emg_right = ts["emg_right"][window_start:window_end]
        times = ts["time"][window_start:window_end]

        x = torch.stack([torch.from_numpy(np.asarray(emg_left)), torch.from_numpy(np.asarray(emg_right))], dim=1)

        start_t = float(times[offset - window_start])
        end_t = float(times[(offset + self.window_length - 1) - window_start])
        y = torch.as_tensor(LabelData.from_keystrokes(self.keystrokes, start_t, end_t).labels)
        return x, y

    @staticmethod
    def collate(samples: Sequence[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
        xs = [s[0] for s in samples]
        ys = [s[1] for s in samples]
        inputs = nn.utils.rnn.pad_sequence(xs)
        targets = nn.utils.rnn.pad_sequence(ys)
        input_lengths = torch.as_tensor([len(x) for x in xs], dtype=torch.int32)
        target_lengths = torch.as_tensor([len(y) for y in ys], dtype=torch.int32)
        return {"inputs": inputs, "targets": targets, "input_lengths": input_lengths, "target_lengths": target_lengths}


def split_recons_paths(data_root: Path, split_yaml: str) -> dict[str, list[Path]]:
    cfg = yaml.safe_load(split_yaml)
    out: dict[str, list[Path]] = {}
    for split in ("train", "val", "test"):
        paths: list[Path] = []
        for item in cfg["dataset"].get(split, []):
            p = data_root / (item["session"] + "_recons_v3.hdf5")
            if not p.exists():
                raise FileNotFoundError(f"Missing recons file: {p}")
            paths.append(p)
        out[split] = paths
    return out


def get_recons_dataloaders(data_root: Path, split_yaml: str, window_length: int = 500, batch_size: int = 32, num_workers: int = 0) -> dict[str, DataLoader]:
    paths = split_recons_paths(data_root, split_yaml)
    train_ds = ConcatDataset([ReconsRawDataset(p, window_length=window_length, stride=None, padding=(10, 2), jitter=True) for p in paths["train"]])
    val_ds = ConcatDataset([ReconsRawDataset(p, window_length=window_length, stride=None, padding=(10, 2), jitter=False) for p in paths["val"]])
    test_ds = ConcatDataset([ReconsRawDataset(p, window_length=window_length, stride=None, padding=(10, 2), jitter=False) for p in paths["test"]])

    print(f"Recons raw dataset | train: {len(paths['train'])} sessions ({len(train_ds)} windows) | val: {len(paths['val'])} sessions ({len(val_ds)} windows) | test: {len(paths['test'])} sessions ({len(test_ds)} windows)")

    persistent = num_workers > 0
    return {
        "train": DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
        "val": DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
        "test": DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
    }


class ReconsRNNEncoder(nn.Module):
    """Bidirectional LSTM encoder for raw reconstructed EMG (62.5 Hz).

    Input shape : (T, N, 2, in_channels)
    Output shape: (T, N, num_classes)
    """

    NUM_BANDS: int = 2

    def __init__(
        self,
        in_channels: int = 16,
        proj_dim: int = 128,
        hidden_size: int = 512,
        num_layers: int = 2,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.input_norm = nn.LayerNorm(in_channels)
        self.proj = nn.Linear(in_channels, proj_dim)
        self.proj_norm = nn.LayerNorm(proj_dim)
        rnn_in = self.NUM_BANDS * proj_dim
        self.rnn = nn.LSTM(
            input_size=rnn_in,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=False,
        )
        self.head = nn.Linear(2 * hidden_size, CHARSET.num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x = self.input_norm(inputs)   # (T, N, 2, in_channels)
        x = self.proj(x)              # (T, N, 2, proj_dim)
        x = self.proj_norm(x)         # (T, N, 2, proj_dim)
        x = x.flatten(start_dim=2)   # (T, N, 2 * proj_dim)
        x = x.contiguous()
        x, _ = self.rnn(x)           # (T, N, 2 * hidden_size)
        x = self.head(x)             # (T, N, num_classes)
        return F.log_softmax(x, dim=-1)


RESULTS_DIR = Path("results")


def make_run_id(model: str, num_channels: int, sampling_rate_hz: int, train_fraction: float) -> str:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pct = int(round(train_fraction * 100))
    return f"{model.upper()}_{num_channels}ch_{sampling_rate_hz}hz_{pct}pct_{ts}"


def append_csv(path: Path, columns: list[str], row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    need_header = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        if need_header:
            writer.writeheader()
        writer.writerow({c: row.get(c, "") for c in columns})


def log_epoch(run_id: str, model: str, epoch: int, train_loss: float, val_loss: float, val_cer: float) -> None:
    append_csv(RESULTS_DIR / f"results_curves_{model.upper()}.csv", ["run_id", "epoch", "train_loss", "val_loss", "val_cer"], {"run_id": run_id, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_cer": val_cer})


def log_summary(run_id: str, model: str, epochs: int, num_channels: int, sampling_rate_hz: int, train_fraction: float, input_type: str, final_train_loss: float, final_val_loss: float, final_val_cer: float, test_cer: float, training_time_sec: float, notes: str = "") -> None:
    append_csv(RESULTS_DIR / f"results_summary_{model.upper()}.csv", ["run_id", "timestamp", "model", "epochs", "num_channels", "sampling_rate_hz", "train_fraction", "input_type", "final_train_loss", "final_val_loss", "final_val_cer", "test_cer", "training_time_sec", "notes"], {"run_id": run_id, "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "model": model.upper(), "epochs": epochs, "num_channels": num_channels, "sampling_rate_hz": sampling_rate_hz, "train_fraction": train_fraction, "input_type": input_type, "final_train_loss": final_train_loss, "final_val_loss": final_val_loss, "final_val_cer": final_val_cer, "test_cer": test_cer, "training_time_sec": training_time_sec, "notes": notes})


def lr_lambda(step: int, warmup_steps: int, total_steps: int, min_lr_ratio: float) -> float:
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    p = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * p))


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.CTCLoss, device: torch.device, scheduler: torch.optim.lr_scheduler.LambdaLR) -> float:
    model.train()
    total = 0.0
    for b in loader:
        x = b["inputs"].to(device)
        y = b["targets"].to(device)
        xlen = b["input_lengths"].to(device)
        ylen = b["target_lengths"].to(device)

        optimizer.zero_grad()
        e = model(x)
        loss = criterion(e, y.transpose(0, 1), xlen, ylen)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()
        total += float(loss.item())

    return total / max(1, len(loader))


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, decoder: CTCGreedyDecoder, preview_limit: int = 0) -> tuple[float, dict[str, float]]:
    model.eval()
    criterion = nn.CTCLoss(blank=CHARSET.null_class, zero_infinity=True)
    metric = CharacterErrorRates()
    total = 0.0
    preview: list[tuple[str, str]] = []

    for b in loader:
        x = b["inputs"].to(device)
        y = b["targets"]
        xlen = b["input_lengths"]
        ylen = b["target_lengths"]

        with torch.backends.cudnn.flags(enabled=False):
            e = model(x)

        loss = criterion(e, y.transpose(0, 1).to(device), xlen.to(device), ylen.to(device))
        total += float(loss.item())

        preds = decoder.decode_batch(e.cpu().numpy(), xlen.numpy())
        y_np = y.numpy()
        ylen_np = ylen.numpy()
        for i, p in enumerate(preds):
            t = LabelData.from_labels(y_np[: int(ylen_np[i]), i])
            metric.update(p, t)
            if preview_limit > 0 and len(preview) < preview_limit:
                preview.append((t.text, p.text))

    if preview:
        print("Sample test predictions:")
        for i, (t, p) in enumerate(preview, 1):
            print(f"  [{i}] true: {t}")
            print(f"      pred: {p}")

    return total / max(1, len(loader)), metric.compute()


@dataclass
class TrainConfig:
    model: str = "rnn"
    epochs: int = 150
    batch_size: int = 32
    num_workers: int = 0
    window_length: int = 500
    proj_dim: int = 64
    hidden_size: int = 256
    num_layers: int = 1
    dropout: float = 0.11750003567529757
    lr: float = 8.215117598353862e-4
    weight_decay: float = 9.35611311371853e-3
    min_lr: float = 1e-5
    warmup_epochs: int = 5
    data_root: Path = Path("data_recons")
    checkpoint_dir: Path = Path("Playground_Kai/checkpoints")
    show_examples: int = 5
    notes: str = "RNN_Recons_Unoptimized"


def resolve_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data_recons").exists():
        return cwd
    for p in cwd.parents:
        if (p / "data_recons").exists():
            return p
    return cwd


def run_training(cfg: TrainConfig) -> None:
    seed_everything(13)
    root = resolve_root()
    data_root = (root / cfg.data_root).resolve() if not cfg.data_root.is_absolute() else cfg.data_root
    checkpoint_dir = (root / cfg.checkpoint_dir).resolve() if not cfg.checkpoint_dir.is_absolute() else cfg.checkpoint_dir
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    loaders = get_recons_dataloaders(data_root, SPLIT_YAML, cfg.window_length, cfg.batch_size, cfg.num_workers)

    model = ReconsRNNEncoder(
        in_channels=16,
        proj_dim=cfg.proj_dim,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
    ).to(device)

    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    criterion = nn.CTCLoss(blank=CHARSET.null_class, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    steps_per_epoch = len(loaders["train"])
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    min_lr_ratio = cfg.min_lr / cfg.lr
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: lr_lambda(s, warmup_steps, total_steps, min_lr_ratio))

    train_frac = len(loaders["train"].dataset) / (len(loaders["train"].dataset) + len(loaders["val"].dataset) + len(loaders["test"].dataset))
    run_id = make_run_id("RNN", 32, 63, train_frac)
    decoder = CTCGreedyDecoder()
    best_cer = float("inf")
    ckpt_path = checkpoint_dir / "best_rnn_recons_raw.pt"

    t0 = time.perf_counter()
    final_train_loss = float("nan")
    final_val_loss = float("nan")
    final_val_cer = float("nan")

    for epoch in range(cfg.epochs):
        e0 = time.perf_counter()
        train_loss = train_one_epoch(model, loaders["train"], optimizer, criterion, device, scheduler)
        val_loss, val_metrics = evaluate(model, loaders["val"], device, decoder, preview_limit=0)
        val_cer = float(val_metrics["CER"])

        final_train_loss = train_loss
        final_val_loss = val_loss
        final_val_cer = val_cer

        log_epoch(run_id, "RNN", epoch + 1, train_loss, val_loss, val_cer)

        if val_cer < best_cer:
            best_cer = val_cer
            torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "best_cer": best_cer}, ckpt_path)
            saved = " [saved best]"
        else:
            saved = ""

        dt = time.perf_counter() - e0
        print(f"Epoch {epoch + 1:3d}/{cfg.epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_CER={val_cer:.2f}% | {dt:.1f}s{saved}")

    if ckpt_path.exists():
        best = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(best["model"])
        print(f"Loaded best checkpoint from epoch {int(best['epoch']) + 1}")

    _, test_metrics = evaluate(model, loaders["test"], device, decoder, preview_limit=cfg.show_examples)

    print("Test metrics:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.2f}%")

    total_time = time.perf_counter() - t0
    log_summary(run_id, "RNN", cfg.epochs, 32, 63, train_frac, "raw_channels", final_train_loss, final_val_loss, final_val_cer, float(test_metrics.get("CER", float("nan"))), total_time, notes=cfg.notes + " recons")

    log_path = checkpoint_dir / "log_model_training.txt"
    with log_path.open("a", encoding="utf-8") as f:
        f.write("\n################### New Entry ###################\n")
        f.write("Model           : rnn\n")
        f.write("Pipeline        : recons-raw (data_recons/, raw channels 16chx2, 62.5 Hz)\n")
        f.write(f"Best val CER    : {best_cer:.2f}%\n")
        for k, v in test_metrics.items():
            f.write(f"Test {k}: {v:.2f}%\n")

    print(f"Training complete in {time.strftime('%H:%M:%S', time.gmtime(total_time))}")
    print(f"Best checkpoint: {ckpt_path}")


In [ ]:
# Build config from embedded hyperparameters
best_hp = yaml.safe_load(BEST_HYPERPARAMS_YAML)
cfg = TrainConfig(
    model="rnn",
    epochs=150,
    batch_size=32,
    num_workers=0,
    window_length=500,
    proj_dim=int(best_hp.get("proj_dim", 64)),
    hidden_size=int(best_hp.get("hidden_size", 256)),
    num_layers=int(best_hp.get("num_layers", 1)),
    dropout=float(best_hp.get("dropout", 0.2)),
    lr=float(best_hp.get("lr", 5e-4)),
    weight_decay=float(best_hp.get("weight_decay", 1e-2)),
    data_root=Path("data_recons"),
    checkpoint_dir=Path("Playground_Kai/checkpoints"),
    show_examples=5,
)
cfg


In [ ]:
# Run training and final test evaluation
run_training(cfg)
